## Creating a complete set of data

### GOALS
* Look at difference between the test and the scraped data month by month (first half of 2020)
* Clean up both sets of data
    * Self scaped modify the date
    * Copy the title when the description is empty
    * Change the date field to an actual date format and not string
    * Make suer columns with both sets are uniform
    * Merge
    * Save

In [1]:
import pandas as pd

In [2]:
scraped_data = pd.read_csv('../data/reuters/business_news_2020_20210416.csv')
kaggle_data = pd.read_csv('../data/kaggle/financial-news-headlines/reuters_headlines.csv')

In [5]:
scraped_data.tail(30)

,Unnamed: 0,title,description,date
26087,26087,"Surveillance in a leafy enclave, Ghosn's Tokyo...",The imposing home where Carlos Ghosn lived for...,Jan 01 2020
26088,26088,Factbox: U.N. sanctions on North Korea current...,The United Nations Security Council has issued...,no_date
26089,26089,China seriously concerned by North Korean warn...,no_description,no_date
26090,26090,Trump orders review of visa program to encoura...,no_description,no_date
26091,26091,Ghosn flight prompts talk of more curbs in Jap...,"Carlos Ghosn's daring flight from Japan, where...",Jan 01 2020
26092,26092,"Surveillance in a leafy enclave, Ghosn's Tokyo...",The imposing home where Carlos Ghosn lived for...,Jan 01 2020
26093,26093,Samsung Electronics chip output at South Korea...,Samsung Electronics partly halted some semico...,Jan 01 2020
26094,26094,U.S. auto safety agency to investigate fatal T...,The fatal Dec. 29 crash of a Tesla Inc vehicle...,Dec 31 2019
26095,26095,Tesla must face lawsuit claiming racism at Cal...,A federal judge rejected Tesla Inc's effort to...,Dec 31 2019
26096,26096,Turkish Airlines says reaches compensation dea...,Turkish Airlines has agreed a compensation dea...,Dec 31 2019


In [6]:
# Trimming to 2020
scraped_data = scraped_data.iloc[:26094]

In [7]:
scraped_data.tail(5)

,Unnamed: 0,title,description,date
26089,26089,China seriously concerned by North Korean warn...,no_description,no_date
26090,26090,Trump orders review of visa program to encoura...,no_description,no_date
26091,26091,Ghosn flight prompts talk of more curbs in Jap...,"Carlos Ghosn's daring flight from Japan, where...",Jan 01 2020
26092,26092,"Surveillance in a leafy enclave, Ghosn's Tokyo...",The imposing home where Carlos Ghosn lived for...,Jan 01 2020
26093,26093,Samsung Electronics chip output at South Korea...,Samsung Electronics partly halted some semico...,Jan 01 2020


In [8]:
scraped_data = scraped_data.drop(['Unnamed: 0'], axis=1)

In [10]:
# I'm sorry for the python sins that I am commiting I just couldn't see how I would be able to do this with zip, apply.
# For this usecase I'm not aware of a 'cythonized' method
for i in range(scraped_data.index[-1] + 1):
    row = scraped_data.iloc[i]
    # Take the date of the previous row -> won't accidentally date something earlier (There was news every day)
    if 'no_date' == row['date']:
        scraped_data.iloc[i]['date'] = scraped_data.iloc[i - 1]['date']
        
    if 'no_description' == row['description']:
        scraped_data.iloc[i]['description'] = row['title']
        
scraped_data.tail(4)

,title,description,date
26090,Trump orders review of visa program to encoura...,Trump orders review of visa program to encoura...,Jan 01 2020
26091,Ghosn flight prompts talk of more curbs in Jap...,"Carlos Ghosn's daring flight from Japan, where...",Jan 01 2020
26092,"Surveillance in a leafy enclave, Ghosn's Tokyo...",The imposing home where Carlos Ghosn lived for...,Jan 01 2020
26093,Samsung Electronics chip output at South Korea...,Samsung Electronics partly halted some semico...,Jan 01 2020


In [11]:
len(scraped_data)

26094

In [14]:
scraped_data.drop_duplicates(inplace=True, ignore_index=True)

In [15]:
len(scraped_data)

26065

In [16]:
len(kaggle_data)

32770

In [17]:
kaggle_data.drop_duplicates(inplace=True, ignore_index=True)

In [18]:
len(kaggle_data) 

32715

The duplicates are slightly concerning. The default setting however is to keep the first occurence (i.e. the most recent date in the case of our dataframes) so there is not risk of look ahead bias. My assumption is that there was a story updated and posted again. Although it is a shame to lose the original reporting of the story I think it is better to consistentantly ignore this. 

The analysis of sentiment is likely to look at several days to weeks.

## Looking at overlap
* Update date field
* Make the columns confirm
* Break down of the overlap between jan-jun 2020

In [19]:
kaggle_data.head(1)

,Headlines,Time,Description
0,TikTok considers London and other locations fo...,Jul 18 2020,TikTok has been in discussions with the UK gov...


In [20]:
kaggle_data.columns = ['title', 'date', 'description']
kaggle_data = kaggle_data[['title', 'description', 'date']]
kaggle_data.head(1)

,title,description,date
0,TikTok considers London and other locations fo...,TikTok has been in discussions with the UK gov...,Jul 18 2020


In [21]:
scraped_data.head(1)

,title,description,date
0,Insurer Hartford to pay $650 million for claim...,Insurer Hartford Financial Services Group said...,Apr 16 2021


In [22]:
scraped_data['date'] = pd.to_datetime(scraped_data['date'])
kaggle_data['date']  = pd.to_datetime(kaggle_data['date'])

In [41]:
# Definitely not the most optimal way to get the date ranges
def get_first_half_of_2020(df):
    first_half = df.loc[(df['date'].dt.year==2020) & (df['date'].dt.month<7)]
    # I have already made sure that there are no duplicates in the data. For the comparison I need to be sure of this
    assert first_half.duplicated().any() == False
    return [first_half.loc[first_half['date'].dt.month==month] for month in range(1,7)]
    
#    return {
#        'jan': first_half.loc[first_half['date'].dt.month==1],
#        'feb': first_half.loc[first_half['date'].dt.month==2],
#        'mar': first_half.loc[first_half['date'].dt.month==3],
#        'apr': first_half.loc[first_half['date'].dt.month==4],
#        'may': first_half.loc[first_half['date'].dt.month==5],
#        'jun': first_half.loc[first_half['date'].dt.month==6]
#    }

In [42]:
scraped_data_time_overlap = get_first_half_of_2020(scraped_data)
kaggle_data_time_overlap = get_first_half_of_2020(kaggle_data)

In [44]:
# scraped_data_time_overlap

## Counts

In [51]:
import calendar

assert(len(scraped_data_time_overlap) == 6)
assert(len(kaggle_data_time_overlap) == 6)

for i in range(6):
    count_scrape = len(scraped_data_time_overlap[i])
    count_kaggle = len(kaggle_data_time_overlap[i])
    difference = abs(count_scrape - count_kaggle)
    
    print(str(calendar.month_abbr[i + 1]) + "\t scrape = {} \t kaggle = {} \t difference = {}".format(count_scrape,count_kaggle, difference))

Jan	 scrape = 1451 	 kaggle = 1125 	 difference = 326
Feb	 scrape = 1539 	 kaggle = 1205 	 difference = 334
Mar	 scrape = 2361 	 kaggle = 1821 	 difference = 540
Apr	 scrape = 2126 	 kaggle = 1654 	 difference = 472
May	 scrape = 1539 	 kaggle = 1190 	 difference = 349
Jun	 scrape = 1678 	 kaggle = 1310 	 difference = 368


So it seems that my data is richer, containing more articles for each of the months. Lets see how this breaks down on an article level